# Byte-Pair Encoding (BPE)

**Companion wiki page:** https://ml-viz.vercel.app/wiki/bpe-tokenization

A pure-Python BPE trainer and tokenizer: learn the merge list from a toy corpus, watch the vocabulary grow, then tokenize unseen words with the learned merges.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['axes.labelcolor'] = '#e2e8f0'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2d3148'
plt.rcParams['grid.color'] = '#2d3148'
np.random.seed(42)

## The training corpus

BPE operates on word frequencies. Each word is split into characters plus an end-of-word marker `</w>` so the algorithm can learn that, e.g., "w" at the end of a word is different from "w" in the middle.

In [ ]:
corpus = {"low": 5, "lower": 2, "newest": 6, "widest": 3}

vocab = {" ".join(word) + " </w>": freq for word, freq in corpus.items()}
for w, f in vocab.items():
    print(f"{f} × {w}")

## Pair counting and merging

Each round: count every adjacent symbol pair (weighted by word frequency), merge the most frequent pair everywhere, record the merge.

In [ ]:
def get_pairs(vocab):
    pairs = {}
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pair = (symbols[i], symbols[i + 1])
            pairs[pair] = pairs.get(pair, 0) + freq
    return pairs

def merge_vocab(pair, vocab):
    bigram = " ".join(pair)
    replacement = "".join(pair)
    return {word.replace(bigram, replacement): freq for word, freq in vocab.items()}

def train_bpe(vocab, num_merges):
    merges = []
    for _ in range(num_merges):
        pairs = get_pairs(vocab)
        if not pairs:
            break
        best = max(pairs, key=pairs.get)
        vocab = merge_vocab(best, vocab)
        merges.append(best)
        print(f"merge {len(merges):2d}: {best} (count {pairs[best]})")
    return merges, vocab

merges, final_vocab = train_bpe(dict(vocab), num_merges=10)
print()
print("final word representations:")
for w, f in final_vocab.items():
    print(f"  {f} × {w}")

## Tokenizing new text

The learned **merge list** *is* the tokenizer: split a new word into characters, then apply the same merges in the same order.

In [ ]:
def tokenize(word, merges):
    symbols = list(word) + ["</w>"]
    for a, b in merges:
        i = 0
        while i < len(symbols) - 1:
            if symbols[i] == a and symbols[i + 1] == b:
                symbols[i:i + 2] = [a + b]
            else:
                i += 1
    return symbols

for word in ["lowest", "newer", "wide"]:
    print(f"{word:8s} → {tokenize(word, merges)}")

Note "lowest": even though it never appeared in training, it decomposes into subwords learned from "low" and "widest"/"newest" — no `[UNK]` needed. This is the OOV robustness that made subword tokenization the standard.

## Vocabulary growth

Each merge adds exactly one token to the vocabulary. Plot how word representations shorten as the vocabulary grows.

In [ ]:
avg_lengths = []
vocab_iter = {" ".join(w) + " </w>": f for w, f in corpus.items()}
total_words = sum(corpus.values())
for k in range(len(merges) + 1):
    avg_len = sum(len(w.split()) * f for w, f in vocab_iter.items()) / total_words
    avg_lengths.append(avg_len)
    if k < len(merges):
        vocab_iter = merge_vocab(merges[k], vocab_iter)

plt.figure(figsize=(8, 4))
plt.plot(avg_lengths, marker='o', color='#6366f1')
plt.xlabel('merges applied'); plt.ylabel('avg tokens per word')
plt.title('BPE: sequences shorten as the vocabulary grows')
plt.grid(alpha=0.3); plt.show()

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: the concept is recapped, the code outline is set, and `# TODO(you)` marks what you fill in. Run the `assert` cell after each — it passes silently when your answer is right.

### Exercise — the first merge, by hand

**Recap:** the first merge is the most frequent adjacent pair across the corpus, weighted by word frequency. With corpus `{"low": 5, "lower": 2, "newest": 6, "widest": 3}`, compute the pair counts on the *initial* character vocabulary and return the winning pair as a tuple like `("a", "b")`.

In [ ]:
def first_merge():
    # TODO(you): build the initial vocab, call get_pairs, and return
    # the most frequent pair (use max with key=).
    ...

fm = first_merge()
fm

In [ ]:
# Run me — passes silently when correct
v0 = {" ".join(w) + " </w>": f for w, f in corpus.items()}
expected = max(get_pairs(v0), key=get_pairs(v0).get)
assert fm is not None and not isinstance(fm, type(Ellipsis)), "fill in the TODO first"
assert tuple(fm) == expected, f"expected the most frequent pair"
print()

<details>
<summary>Solution</summary>

```python
def first_merge():
    v0 = {" ".join(w) + " </w>": f for w, f in corpus.items()}
    pairs = get_pairs(v0)
    return max(pairs, key=pairs.get)
```
</details>